In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import os
os.environ["DATA_BASE_PATH"] = "/new_data/knowledge_rh"

In [ ]:
from datasets import load_dataset, concatenate_datasets
from openai import OpenAI

endpoint = f"http://localhost:8000/v1"
openai_api_key = "EMPTY"
openai_api_base = endpoint

client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)
teacher_model = client.models.list().data[0].id

In [17]:
from sdg_hub.flow import Flow
from sdg_hub.pipeline import Pipeline
from sdg_hub.sdg import SDG
from sdg_hub.prompts import PromptRegistry
from sdg_hub.blocks import BlockRegistry, Block
from transformers import AutoTokenizer
import re
from typing import List
from datasets import load_dataset, Dataset

tokenizer = AutoTokenizer.from_pretrained("nvidia/Llama-3_3-Nemotron-Super-49B-v1")

@PromptRegistry.register("nvidia/Llama-3_3-Nemotron-Super-49B-v1")
def nemotron_chat_template():
    return """{{- bos_token }}
{{- "<|start_header_id|>system<|end_header_id|>\n\n" }}detailed thinking on{{- "<|eot_id|>" }}
{%- for message in messages %}
  {%- if message['role'] == 'assistant' and '</think>' in message['content'] %}
    {%- set content = message['content'].split('</think>')[-1].lstrip() %}
  {%- else %}
    {%- set content = message['content'] %}
  {%- endif %}
  {{- '<|start_header_id|>' + message['role'] + '<|end_header_id|>\n\n' + content | trim + '<|eot_id|>' }}
{%- endfor %}
{%- if add_generation_prompt %}
  {{- '<|start_header_id|>assistant<|end_header_id|>\n\n' }}
{%- endif %}"""

ds = load_dataset('json', data_files="seed_data.jsonl", split="train")
@BlockRegistry.register("PostProcessThinkingBlock")
class PostProcessThinkingBlock(Block):
    def __init__(self, block_name: str, column_name: str) -> None:
        super().__init__(block_name=block_name)  
        self.column_name = column_name
    
    
    def generate(self, samples: Dataset):
        def post_process_thinking(x):
            if '</think>' in x[self.column_name]:
                x[self.column_name] = x[self.column_name].split('</think>')[-1].lstrip()
            return x
        samples = samples.map(post_process_thinking)
        return samples

@BlockRegistry.register("RegexParserBlock")
class RegexParserBlock(Block):
    def __init__(self, block_name: str, column_name: str, parsing_pattern: str="", parser_cleanup_tags: List[str]=[], output_cols: List[str]=[]) -> None:
        super().__init__(block_name=block_name)
        self.column_name = column_name
        self.parsing_pattern = parsing_pattern
        self.parser_cleanup_tags = parser_cleanup_tags
        self.output_cols = output_cols

    def generate(self, samples: Dataset):
        
        if self.parsing_pattern:
            new_data = []
            for sample in samples:
                parsed_outputs = self._parse(sample[self.column_name])
                max_length = max(len(value) for value in parsed_outputs.values())
                for values in zip(*(lst[:max_length] for lst in parsed_outputs.values())):
                    new_data.append({**sample, **dict(zip(parsed_outputs.keys(), values))})
            samples = Dataset.from_list(new_data)
        if self.parser_cleanup_tags:
            for clean_tag in self.parser_cleanup_tags:
               samples = samples.map(lambda x: {column_name: x[column_name].replace(clean_tag, "") for column_name in self.output_cols})
        return samples

    def _parse(self, generated_string):      
        pattern = re.compile(self.parsing_pattern, re.DOTALL)
        all_matches = pattern.findall(generated_string)
        matches = {column_name: [] for column_name in self.output_cols}
        if all_matches and isinstance(all_matches[0], tuple):
            for match in all_matches:
                for column_name, value in zip(self.output_cols, match):
                    value = value.strip()
                    # for clean_tag in self.parser_cleanup_tags:
                    #     value = value.replace(clean_tag, "")
                    matches[column_name].append(value)
        else:
            matches[self.output_cols[0]] = (
                [match.strip() for match in all_matches] if all_matches else []
            )
        return matches


knowledge_agentic_pipeline = "flows/synth_knowledge_reasoning_nemotron_super_49b_summary_diversity.yaml"
flow_cfg = Flow(client).get_flow_from_file(knowledge_agentic_pipeline)
sdg = SDG(
    [Pipeline(flow_cfg)],
    num_workers=1,
    batch_size=1,
    save_freq=1000,
)

In [ ]:
generated_data = sdg.generate(ds.select([3]), checkpoint_dir="Tmp")

In [ ]:
idx = 2
print("### Raw Document")
print(generated_data['raw_document'][idx])
print("### Generated Summary Instruction")
print(generated_data['summary_instruction'][idx])
print("### Generated Document Summary")
print(generated_data['document'][idx])

### Raw Document
```text
" It's Time To Keelhaul U-Haul!", Jeffrey Goldberg, 1999.
 It's Time To Keelhaul U-Haul! 

         Like all superheroes worthy of the title, the Shopping Avenger has an Achilles' heel. In the case of the Shopping Avenger, his Achilles' heel is not animal, vegetable, or mineral but something less tangible. 

         An explanation: Last week, the magazine you are currently reading forced the Shopping Avenger at gunpoint to read a series of treacle-filled self-help books, and then to . The Shopping Avenger, who can withstand radiation, extreme heat and cold, hail, bear attacks, and Eyes Wide Shut , almost succumbed to terminal jejuneness after reading these books. Except for one thing: One of the books, The Art of Happiness , which collects and simplifies the Dalai Lama's philosophy, got the Shopping Avenger to thinking. This, in a way, is the Shopping Avenger's Achilles' heel: thinking. Perhaps it is wrong, the Shopping Avenger thought, to complain about the petty insults and inconveniences of life in the materialistic '90s. The Shopping Avenger felt that perhaps he should counsel those who write seeking help to meditate, to accept bad service the way one accepts the change of seasons, and to extend a compassionate hand of forgiveness to those who provide poor customer care. 

         But then the Shopping Avenger sat down, and the feeling passed. 

         The Shopping Avenger does not make light of the Dalai Lama or of the notion that there is more to life than the impatient acquisition of material goods. If the Shopping Avenger were not, for a superhero, extremely nonjudgmental--as opposed to his alter ego, who is considered insufferably judgmental by his alter ego's wife--the Shopping Avenger would tell the occasional correspondent to let go of his petty grievance and get a life. 

         But the Shopping Avenger also believes that the Dalai Lama has never tried to rent a truck from U-Haul. If he had tried to rent from U-Haul, he never would have escaped from Tibet. (For the complete back story, see "Shopping Avenger" column and one.) 

         The complaints about U-Haul's nonreservation reservation policy continue to pour in through the electronic mail. One correspondent, B.R., wrote in with this cautionary tale: "Last weekend, I went to San Francisco to help my brother and his family move into their first house. My brother had reserved a moving truck with U-Haul for the big day. I warned my brother about U-Haul's 'not really a reservation per se' policy that I learned from the Shopping Avenger. He didn't believe such a thing would happen to him, so he didn't act on my warning." 

         B.R. continues--as if you don't know what happened already--"I went to U-Haul with my brother to get our 'reserved' truck. The store had many customers standing around looking frustrated. When we got to the front of the line, the clerk informed us that our 'reserved' truck had not yet been returned. We asked if we could rent one of the many trucks sitting idle in the parking lot. The clerk laughed and said the keys to those trucks were lost." 

         B.R. and his chastened brother--the Shopping Avenger is resisting the urge to gloat--went to Ryder. "Ryder had a truck available for us. The gentleman who helped us at Ryder said Ryder prides itself on being everything U-Haul is not." 

         The Shopping Avenger has still not received a call from U-Haul spokeswoman Johna Burke explaining why U-Haul refuses to provide trucks to people who reserve trucks, but the Shopping Avenger is pleased to note that several correspondents have written in over the past month saying that, based on what they have read in this column, they will be taking their business to Ryder or Budget or elsewhere. 

         The Shopping Avenger will undoubtedly return to the sorry state of affairs at U-Haul in the next episode, but now on to this month's airline debacle. 

         Before we begin, though, the Shopping Avenger nearly forgot to announce the winner of last month's contest, in which readers were asked to answer the question, "What's the difference between pests and airlines?" 

         The winner is one Tom Morgan, who wrote, "You can hire someone to kill pests." Tom is the winner of a year's supply of Turtle Wax, and he will receive his prize just as soon as the Shopping Avenger figures out how much Turtle Wax actually constitutes a year's supply. The new contest question: How much Turtle Wax comprises a year's supply of Turtle Wax? 

         This month's airline in the spotlight is Southwest. Loyal readers will recall that last month the Shopping Avenger praised Southwest Airlines for its "sterling" customer service. This brought forth a small number of articulate dissensions. The most articulate, and the most troubling, came from M., who wrote, "Last year, flying from Baltimore to Chicago with my entire family (two really little kids included), we set down at Midway in a rainstorm. And waited for our bags. And waited for bags. And waited for bags." 

         An hour later, M. says, the bags showed up, "soaked through. We took them to baggage services at SW and were faced with the most complicated, unclear, and confusing mechanism for filing a claim we experienced flyers have ever seen." 

         When they arrived at their destination, M. and her family made a terrible discovery, "We discovered that our clothes were soaked through--the top clothes were so wet that the dye had bled through down to the lower levels, destroying lots of other clothes. Obviously, our bags had just been sitting out on the runway in the rain. To this day, I've never heard a thing from SW, despite calls and letters." 

         This, of course, is where Shopping Avenger steps in. Shopping Avenger knows that Southwest is different from the average airline, in that it doesn't go out of its way to infuriate its paying customers (see: ), so I expected a quick and generous resolution to M.'s problem. 

         What I got at first, though, was a load of corporate hoo-ha. 

         "The airline's policy, which is consistent with all contracts of carriage at all airlines, requires that passengers file a report in person for lost or damaged luggage within four hours of arrival at their destination," a Southwest spokeswoman, Linda Rutherford, e-mailed me. "[M.] indicates she called for a few days, but did not file a report in person until April 12--three days later. Southwest, as a courtesy, took her report anyway and asked for follow up information and written inventory of the damage." Rutherford said that M. should have submitted detailed receipts and photographs of the damage in order to make a claim. 

         Harrumph, the Shopping Avenger says. It is a bad hair day at Southwest when its officials defend themselves by comparing their airline to other airlines. I forwarded this message to M., who replied: 

         "Wow. Well, of course I didn't file it at the airport on the 9 th because I didn't know the clothes were ruined at the airport. I didn't know until I opened the baggage at my hotel and saw the ruined stuff. (And it's worth noting that we had already waited for about an hour for our luggage with two little kids and impatient in-laws nipping at our heels.)" 

         She goes on, "I did call that evening ... and was told that that sufficed. This is the first time I've been told that I had to file a complaint in person within four hours. ... When I filed on the 12 th , I was never told that I needed any receipts or photos or other type of documentation. The baggage folks seemed pretty uninterested in all of this. ... They know that the type of 'evidence' they want is impossible to obtain. They also know that on April 9 they screwed up the luggage retrieval and left bags out in the rain a long time." 

         Southwest's response actually served to anger M. more than the original problem. "Before, they had a mildly annoyed but loyal customer (who would have been placated by an apology and thrilled with some modest token of their regret). Now they have a pissed-off customer." 

         Things do look bad for Southwest, don't they? The Shopping Avenger sent M.'s response to Rutherford, who e-mailed back saying she thought the Shopping Avenger was asking for "policy information." The Shopping Avenger e-mailed back again, stressing to Rutherford that the Great Court of Consumer Justice would, if this case were brought to trial, undoubtedly find for the plaintiff (the Shopping Avenger serves as prosecutor, judge, and jury in the Great Court of Consumer Justice--defendants are represented by the president of U-Haul), and that Southwest was precipitously close to feeling the sword of retribution at its neck. 

         But then she came through, provisionally, "Yep, you can be sure if [M.] will call me we will get everything squared away. I'm sorry it's taken this long for her to get someone who can help, but we will take care of it from here." 

         Stay tuned, shoppers, to hear whether Southwest makes good it promise to compensate M. and apologize to her for her troubles. 

         The story of M. reminds the Shopping Avenger of a central truth of consumer service: It's not the crime, it's the cover-up. 

         Take the case of K., who found himself waiting in vain for Circuit City to repair his television. Televisions break, even 1-year-old televisions, as is the case with K's. But Circuit City, where he bought the television, gave him a terrible runaround. The Shopping Avenger dispatched his sidekick, Tad the Deputy Avenger, to get to the bottom of K.'s story. This is what he found: K. grew concerned, Tad the Deputy Avenger reports, after his television had been in the Circuit City shop for a week. When he called, he was told to "check back next week." When he asked if someone from the store could call him with more information, he was refused. Weeks went by. When K. told one Circuit City employee that he really would like to get his television back, the employee, K. says, asked him, "Don't you have another television in your house?" 

         More than a month later--after hours and hours and hours of telephone calls and days missed at work--K. received his television back. 

         Mistakes happen, but not, Tad the Deputy Avenger found out, at Circuit City. The case, K. was told by a Circuit City official, was "handled perfectly." Another official, Morgan Stewart in public relations, assured Deputy Avenger Tad that "We got to be a big and successful company by treating customers better than the other guy." The Shopping Avenger and his loyal sidekick would like to hear from other Circuit City customers: Does Circuit City, in fact, treat its customers better than the other guy? 

         Stay tuned for answers. And next month, a Shopping Avenger clergy special: TWA screws with a Hasidic rabbi's travel plans, leaving the rabbi's wife crying at the airport. Find out if the Shopping Avenger can save TWA from certain heavenly punishment, in the next episode. 

                                            Got a consumer score you want settled? Send e-mail to shoppingavenger@slate.com.
```
### Generated Summary Instruction
```text
**Summarize for a college audience** focusing on the intersection of consumer rights and corporate accountability.
```
### Generated Document Summary
```text
**Summary: "It's Time To Keelhaul U-Haul!" and the Intersection of Consumer Rights and Corporate Accountability**  

Jeffrey Goldberg’s 1999 column, "It's Time To Keelhaul U-Haul!", features "The Shopping Avenger," a satirical consumer advocate, exposing corporate failures in honoring commitments and addressing grievances. The piece highlights three cases illustrating the tension between consumer rights and corporate accountability:  

1. **U-Haul’s Deceptive Reservation Policy**  
   U-Haul’s practice of accepting "reservations" without guaranteeing vehicle availability is critiqued through a reader’s experience. Despite a confirmed reservation, the company denied access to a truck, citing unreturned vehicles and "lost" keys for idle trucks. The Shopping Avenger contrasts this with Ryder’s reliability, noting that readers began boycotting U-Haul in response to such practices. The incident underscores how corporations may prioritize operational convenience over contractual obligations, eroding trust.  

2. **Southwest Airlines’ Baggage Fiasco and Bureaucratic Deflection**  
   A family’s luggage was left in the rain during a Southwest flight, destroying clothing. Southwest initially blamed the passenger for not filing a claim in person within four hours—a policy the family was never informed of. The Shopping Avenger intervened, prompting a spokesperson to eventually promise resolution, but not before the airline’s defensive response exacerbated the issue. This case demonstrates how rigid adherence to policies can override customer fairness and how advocacy can force accountability.  

3. **Circuit City’s Neglect and Denial**  
   A customer’s television repair was delayed for over a month, with Circuit City staff showing indifference (e.g., asking if the customer had another TV). Despite the company’s claim of "perfect" handling, the Shopping Avenger challenged this narrative, inviting public testimony to counter corporate self-exoneration. The example illustrates how companies may dismiss consumer complaints while touting superior service.  

**Analysis for a College Audience**  
These cases illustrate systemic gaps in corporate accountability:  
- **Deceptive Practices**: U-Haul’s reservation policy exemplifies how companies may exploit legal loopholes to prioritize profits over transparency.  
- **Deflective Accountability**: Southwest and Circuit City’s initial responses—blaming policy or the customer—reflect a common corporate strategy to shift responsibility away from operational failures.  
- **Power of Advocacy**: The Shopping Avenger’s interventions highlight how public pressure and media scrutiny can compel corporations to address grievances they might otherwise ignore.  

The column argues that consumer rights are often compromised not by mistakes alone but by corporations’ refusal to acknowledge and rectify them transparently. It emphasizes the importance of vigilant advocacy in holding companies accountable, particularly when legal or policy technicalities are used to evade responsibility. For colleges, this raises questions about ethical business practices, the role of consumer activism, and the need for regulatory oversight to protect rights in a market-driven economy.  

**Key Takeaway**  
The intersection of consumer rights and corporate accountability hinges on transparency, empathy, and willingness to resolve grievances fairly. When companies prioritize bureaucratic self-protection over customer justice, consumer advocacy becomes critical in restoring balance—a dynamic relevant to ongoing debates about regulation, corporate ethics, and the leverage of public scrutiny.
```

In [ ]:
idx = 2
print("### Question")
print(generated_data[idx]['question'])
print("### Generated Answer")
print(generated_data[idx]['response'])

### Question
```text
Apply the framework of "deflective accountability" and "advocacy-driven resolution" to a hypothetical scenario involving a pharmaceutical company’s handling of drug side-effect complaints. How might regulatory oversight influence the dynamic between corporate responsibility and consumer protection in this context?  
```
### Generated Answer
```text
<think>
Okay, let's tackle this question. First, I need to understand what "deflective accountability" and "advocacy-driven resolution" mean based on the document. From the examples, deflective accountability is when companies blame policies, customers, or other factors to avoid taking responsibility. Advocacy-driven resolution happens when external pressure, like media or consumer groups, forces the company to fix the issue.

The question is asking to apply these concepts to a hypothetical pharmaceutical scenario. Let me think of a plausible case. Maybe a drug has side effects, and the company initially denies responsibility or blames the patient. Then, advocacy groups step in, leading to a resolution.

Now, how does regulatory oversight play into this? The document mentions that advocacy can force accountability, but regulations could either support or hinder this. If there are strong regulations, maybe the company can't deflect as easily. But if regulations are weak, the company might rely more on deflective strategies until advocacy pushes them.

I should structure the answer with the hypothetical scenario first, then apply the two frameworks, and finally discuss regulatory influence. Make sure to tie it back to the document's analysis, like how deflective accountability works and how advocacy drives change. Also, mention the Key Takeaway about transparency and empathy.

Wait, the answer needs to be concise. Let me outline:

1. Hypothetical: Pharma company's drug causes side effects. They blame pre-existing conditions or say it's rare, ignoring complaints.
2. Deflective Accountability: Company uses medical jargon, cites approval processes, shifts blame.
3. Advocacy-Driven Resolution: Patient groups, media expose the issue, leading to investigations and recalls.
4. Regulatory Oversight: Strong regulations (like mandatory reporting) would reduce deflective tactics. Weak regulations allow more deflection until advocacy steps in. Regulatory bodies can enforce transparency, balance corporate and consumer interests.

Check the document's examples: U-Haul's policy, Southwest's blame-shifting. The analysis part talks about systemic gaps and the role of advocacy. The Key Takeaway emphasizes transparency and advocacy's role when companies prioritize self-protection.

Make sure the answer references these elements. Also, the implications of challenging the assumption in the previous example might not be relevant here, but the structure of applying the frameworks and discussing regulation is key.

Avoid going off-track. Keep each part clear. Use terms from the document like "deflective accountability," "advocacy-driven resolution," and mention how regulations affect the dynamic, possibly by either enabling or mitigating the company's deflective strategies.
</think>


**Hypothetical Scenario**: A pharmaceutical company, "MediCorp," releases a drug with undisclosed severe side effects. When patients report issues, MediCorp attributes them to "pre-existing conditions" or claims the complaints are statistically rare, citing FDA approval as justification. Initially, they refuse compensation or recalls, blaming "isolated incidents" and emphasizing the drug’s "proven efficacy."  

- **Deflective Accountability**: MediCorp employs deflective strategies by (1) blaming patients’ health histories, (2) invoking regulatory approval to legitimize inaction, and (3) downplaying severity through statistical minimization. This mirrors Southwest’s policy-based deflection and Circuit City’s denial, prioritizing reputational protection over accountability.  

- **Advocacy-Driven Resolution**: A patient advocacy group, aided by media exposure, compiles evidence of widespread harm and organizes a boycott. Public pressure forces MediCorp to establish a compensation fund and recall the drug, illustrating how external scrutiny overrides corporate resistance, as seen in U-Haul’s boycott response.  

**Regulatory Oversight Influence**:  
- **Strong Oversight**: Mandatory adverse-event reporting laws and independent review boards could prevent deflective tactics by forcing transparency (e.g., public side-effect databases). Regulations might also impose fines for non-disclosure, aligning corporate incentives with consumer safety.  
- **Weak Oversight**: Without such measures, MediCorp could delay action until advocacy intervenes, replicating the "systemic gaps" noted in the column. Effective regulation would reduce reliance on advocacy by preemptively ensuring accountability, balancing corporate autonomy with consumer protection.  

This dynamic reflects the document’s thesis that corporate accountability often requires either robust regulation or public advocacy to counter self-serving practices, particularly when transparency and empathy are absent.
```

In [5]:
from utils import create_training_mix
from transformers import AutoTokenizer
from datasets import load_dataset, concatenate_datasets
 

data_base_path = os.environ["DATA_BASE_PATH"]
# Load tokenizer for pre-training formatting
tokenizer = AutoTokenizer.from_pretrained("nvidia/Llama-3.1-Nemotron-Nano-8B-v1")


# Load reasoning dataset from nemotron super 49b
quality_reasoning_ds = load_dataset("json", data_dir=f"{data_base_path}/quality/synth_knowledge_reasoning_nemotron_super_49b_summary_diversity/", split="train")
print(quality_reasoning_ds)
# Create reasoning training mix
quality_reasoning_ds = create_training_mix(quality_reasoning_ds, tokenizer, 'on', create_summary=False)

Dataset({
    features: ['document_outline', 'raw_document', 'icl_document', 'icl_query_1', 'icl_response_1', 'icl_query_2', 'icl_response_2', 'icl_query_3', 'icl_response_3', 'domain', 'summary_instruction', 'document', 'question', 'response'],
    num_rows: 35589
})


In [15]:
quality_reasoning_ds

Dataset({
    features: ['messages', 'metadata', 'id', 'unmask'],
    num_rows: 35589
})

In [6]:
# Load non-reasoning dataset from nemotron super 49b
quality_non_reasoning_ds = load_dataset("json", data_dir=f"{data_base_path}/quality/knowledge1.25_nemotron_super_49b/", split="train")
quality_non_reasoning_ds = quality_non_reasoning_ds.filter(lambda x: x['score'] == '2' and x['judgment'] == 'YES')
print(quality_non_reasoning_ds)

# Create non-reasoning training mix
quality_non_reasoning_ds = create_training_mix(quality_non_reasoning_ds, tokenizer, 'off', create_summary=False).shuffle(seed=894375).select(range(30000))

Dataset({
    features: ['document_outline', 'raw_document', 'icl_document', 'icl_query_1', 'icl_response_1', 'icl_query_2', 'icl_response_2', 'icl_query_3', 'icl_response_3', 'domain', 'dataset_type', 'document', 'qa_pairs', 'question', 'response', 'explanation', 'judgment', 'feedback', 'score', 'rating'],
    num_rows: 72104
})


In [10]:
# Load nemotron replay buffer
nemotron_ds_replay_buffer = load_dataset("json", data_files=f"{data_base_path}/quality/training_mix/nemotron_replay_buffer_data.jsonl", split="train")


# Concatenate training mix with replay buffer
training_mix = concatenate_datasets([quality_reasoning_ds, quality_non_reasoning_ds, nemotron_ds_replay_buffer.shuffle(seed=894375).select(range(len(quality_reasoning_ds)))])

training_mix_only_reasoning = concatenate_datasets([quality_reasoning_ds, nemotron_ds_replay_buffer.shuffle(seed=894375).select(range(len(quality_reasoning_ds)))])

print(training_mix)
training_mix.to_json(f"{data_base_path}/quality/training_mix/quality_reasoning_mix_with_summary_diversity_with_30k_non_reasoning.jsonl", orient='records', lines=True)
# training_mix_only_reasoning.to_json(f"{data_base_path}/quality/training_mix/quality_reasoning_mix_only_reasoning.jsonl", orient='records', lines=True)

Dataset({
    features: ['messages', 'metadata', 'id', 'unmask'],
    num_rows: 101178
})


Creating json from Arrow format: 100%|██████████| 102/102 [00:19<00:00,  5.34ba/s]


1997926851

In [18]:
f"{data_base_path}/quality/training_mix/quality_reasoning_mix_with_summary_diversity.jsonl"

'/new_data/knowledge_rh/quality/training_mix/quality_reasoning_mix_with_summary_diversity.jsonl'

In [8]:
tulu_v3 = load_dataset("allenai/tulu-3-sft-mixture", split="train")
training_mix_only_reasoning = training_mix_only_reasoning.remove_columns(['metadata', 'id', 'unmask'])
tulu_v3_small = tulu_v3.select(range(20000)).remove_columns( ['id', 'source'])
tulu_v3_small = tulu_v3_small.map(lambda x: {'messages': [{'content': 'detailed thinking off', 'role': 'system'}] + x['messages']}, batched=False)
training_mix_only_reasoning_tulu = concatenate_datasets([training_mix_only_reasoning, tulu_v3_small])
training_mix_only_reasoning_tulu.to_json(f"{data_base_path}/quality/training_mix/quality_reasoning_mix_only_reasoning_tulu.jsonl", orient='records', lines=True)

training_mix_tulu = concatenate_datasets([training_mix.remove_columns(['metadata', 'id', 'unmask']), tulu_v3_small])
training_mix_tulu.to_json(f"{data_base_path}/quality/training_mix/quality_reasoning_mix_with_summary_diversity_tulu.jsonl", orient='records', lines=True)

Creating json from Arrow format: 100%|██████████| 122/122 [00:06<00:00, 18.32ba/s]


855573238

In [9]:
training_mix_tulu

Dataset({
    features: ['messages'],
    num_rows: 121178
})